In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
def dfs_por_ano(url):
    urls = [url.replace('{ano_ref}', str(ano)) for ano in range(2012, 2026)]
    dfs = [pd.read_csv(url) for url in urls]
    return pd.concat(dfs)

In [3]:
url_pessoas = 'https://aplicacoes.mds.gov.br/sagi/servicos/misocial/?fl=codigo_ibge%2Canomes_s%20cadun_qtd_pessoas_cadastradas_i&fq=cadun_qtd_pessoas_cadastradas_i%3A*&q=*%3A*&rows=100000&sort=anomes_s%20desc%2C%20codigo_ibge%20asc&wt=csv&fq=anomes_s:{ano_ref}*'

In [4]:
dfs_pessoas = dfs_por_ano(url_pessoas)

In [5]:
dfs_pessoas.head()

,codigo_ibge,anomes_s,cadun_qtd_pessoas_cadastradas_i
0,110001,201212,14076
1,110002,201212,37567
2,110003,201212,3313
3,110004,201212,29459
4,110005,201212,7918


In [6]:
dfs_pessoas.shape

(891109, 3)

In [7]:
dfs_pessoas.dtypes

codigo_ibge                        int64
anomes_s                           int64
cadun_qtd_pessoas_cadastradas_i    int64
dtype: object

In [8]:
dfs_pessoas.isna().sum()

codigo_ibge                        0
anomes_s                           0
cadun_qtd_pessoas_cadastradas_i    0
dtype: int64

In [9]:
url_familias = 'https://aplicacoes.mds.gov.br/sagi/servicos/misocial/?fl=codigo_ibge%2Canomes_s%20cadun_qtd_familias_cadastradas_i&fq=cadun_qtd_familias_cadastradas_i%3A*&q=*%3A*&rows=100000&sort=anomes_s%20desc%2C%20codigo_ibge%20asc&wt=csv&fq=anomes_s:{ano_ref}*'

In [10]:
dfs_familias = dfs_por_ano(url_familias)

In [11]:
dfs_familias.head()

,codigo_ibge,anomes_s,cadun_qtd_familias_cadastradas_i
0,110001,201212,4116
1,110002,201212,10775
2,110003,201212,993
3,110004,201212,8516
4,110005,201212,2259


In [12]:
dfs_familias.shape

(891109, 3)

In [13]:
dfs_familias.dtypes

codigo_ibge                         int64
anomes_s                            int64
cadun_qtd_familias_cadastradas_i    int64
dtype: object

In [14]:
dfs_familias.isna().sum()

codigo_ibge                         0
anomes_s                            0
cadun_qtd_familias_cadastradas_i    0
dtype: int64

In [15]:
df_juntado = (
    dfs_familias.merge(
        dfs_pessoas,
        on=['codigo_ibge', 'anomes_s']
    )
    .assign(
        mes = lambda x: x['anomes_s'] % 100,
        ano = lambda x: x['anomes_s'] // 100,
    )
    .drop(columns=['anomes_s'])
    .rename(columns={
        'cadun_qtd_familias_cadastradas_i': 'qtde_familias',
        'cadun_qtd_pessoas_cadastradas_i': 'qtde_pessoas',
    })
)

In [16]:
df_juntado.head()

,codigo_ibge,qtde_familias,qtde_pessoas,mes,ano
0,110001,4116,14076,12,2012
1,110002,10775,37567,12,2012
2,110003,993,3313,12,2012
3,110004,8516,29459,12,2012
4,110005,2259,7918,12,2012


In [17]:
df_cod_cidades = pd.read_excel('../../compartilhado/RELATORIO_DTB_BRASIL_2025_MUNICIPIOS.ods')

In [18]:
df_cod_cidades.head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,379,1100379,Alto Alegre dos Parecis
2,11,Rondônia,1101,Porto Velho,110002,Ariquemes,403,1100403,Alto Paraíso
3,11,Rondônia,1102,Ji-Paraná,110004,Ji-Paraná,346,1100346,Alvorada D'Oeste
4,11,Rondônia,1101,Porto Velho,110002,Ariquemes,23,1100023,Ariquemes


In [19]:
df_cod_cidades = df_cod_cidades[['Nome_UF', 'Código Município Completo', 'Nome_Município']]

In [20]:
df_cod_cidades.head()

,Nome_UF,Código Município Completo,Nome_Município
0,Rondônia,1100015,Alta Floresta D'Oeste
1,Rondônia,1100379,Alto Alegre dos Parecis
2,Rondônia,1100403,Alto Paraíso
3,Rondônia,1100346,Alvorada D'Oeste
4,Rondônia,1100023,Ariquemes


In [21]:
df_cod_cidades.isna().sum()

Nome_UF                      0
Código Município Completo    0
Nome_Município               0
dtype: int64

In [22]:
df_cod_cidades.dtypes

Nome_UF                        str
Código Município Completo    int64
Nome_Município                 str
dtype: object

In [23]:
df_cod_cidades = (
    df_cod_cidades.rename(columns={
        'Nome_UF': 'uf',
        'Nome_Município': 'municipio',
    })
    .assign(
        codigo_ibge = lambda x: x['Código Município Completo'] // 10
    )
    .drop(columns='Código Município Completo')
)

In [24]:
df_cod_cidades.head()

,uf,municipio,codigo_ibge
0,Rondônia,Alta Floresta D'Oeste,110001
1,Rondônia,Alto Alegre dos Parecis,110037
2,Rondônia,Alto Paraíso,110040
3,Rondônia,Alvorada D'Oeste,110034
4,Rondônia,Ariquemes,110002


In [25]:
df_juntado = df_juntado.merge(df_cod_cidades, on='codigo_ibge')

In [26]:
df_juntado.head()

,codigo_ibge,qtde_familias,qtde_pessoas,mes,ano,uf,municipio
0,110001,4116,14076,12,2012,Rondônia,Alta Floresta D'Oeste
1,110002,10775,37567,12,2012,Rondônia,Ariquemes
2,110003,993,3313,12,2012,Rondônia,Cabixi
3,110004,8516,29459,12,2012,Rondônia,Cacoal
4,110005,2259,7918,12,2012,Rondônia,Cerejeiras


In [27]:
gdf_ufs = gpd.read_file('../../compartilhado/BR_UF_2025')

In [28]:
gdf_ufs.head()

,CD_UF,NM_UF,SIGLA_UF,CD_REGIAO,NM_REGIAO,SIGLA_RG,AREA_KM2,geometry
0,43,Rio Grande do Sul,RS,4,Sul,S,281707.150,"MULTIPOLYGON (((-53.52154 -33.25881, -53.51825..."
1,35,São Paulo,SP,3,Sudeste,SE,248219.485,"MULTIPOLYGON (((-48.03575 -25.35712, -48.03607..."
2,32,Espírito Santo,ES,3,Sudeste,SE,46074.440,"MULTIPOLYGON (((-40.88385 -21.16198, -40.88384..."
3,33,Rio de Janeiro,RJ,3,Sudeste,SE,43750.424,"MULTIPOLYGON (((-44.72025 -23.35934, -44.72029..."
4,41,Paraná,PR,4,Sul,S,199293.571,"MULTIPOLYGON (((-48.40723 -25.84254, -48.40732..."


In [29]:
gdf_ufs.isna().sum()

CD_UF        0
NM_UF        0
SIGLA_UF     0
CD_REGIAO    0
NM_REGIAO    0
SIGLA_RG     0
AREA_KM2     0
geometry     0
dtype: int64

In [30]:
gdf_ufs.dtypes

CD_UF             str
NM_UF             str
SIGLA_UF          str
CD_REGIAO         str
NM_REGIAO         str
SIGLA_RG          str
AREA_KM2      float64
geometry     geometry
dtype: object

In [31]:
gdf_ufs = (
    gdf_ufs[['NM_UF', 'SIGLA_UF', 'NM_REGIAO']]
    .rename(columns={
        'NM_UF': 'uf',
        'SIGLA_UF': 'sigla_uf',
        'NM_REGIAO': 'regiao'
    })
)

In [32]:
gdf_ufs.columns

Index(['uf', 'sigla_uf', 'regiao'], dtype='str')

In [33]:
df_juntado = df_juntado.merge(
    gdf_ufs,
    on='uf'
)

In [34]:
df_juntado.head()

,codigo_ibge,qtde_familias,qtde_pessoas,mes,ano,uf,municipio,sigla_uf,regiao
0,110001,4116,14076,12,2012,Rondônia,Alta Floresta D'Oeste,RO,Norte
1,110002,10775,37567,12,2012,Rondônia,Ariquemes,RO,Norte
2,110003,993,3313,12,2012,Rondônia,Cabixi,RO,Norte
3,110004,8516,29459,12,2012,Rondônia,Cacoal,RO,Norte
4,110005,2259,7918,12,2012,Rondônia,Cerejeiras,RO,Norte


In [35]:
df_juntado.info()

<class 'pandas.DataFrame'>
RangeIndex: 891109 entries, 0 to 891108
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   codigo_ibge    891109 non-null  int64
 1   qtde_familias  891109 non-null  int64
 2   qtde_pessoas   891109 non-null  int64
 3   mes            891109 non-null  int64
 4   ano            891109 non-null  int64
 5   uf             891109 non-null  str  
 6   municipio      891109 non-null  str  
 7   sigla_uf       891109 non-null  str  
 8   regiao         891109 non-null  str  
dtypes: int64(5), str(4)
memory usage: 88.0 MB


In [37]:
df_juntado.to_parquet('qtde_pessoas_familias_cadunico.parquet', index=False)